# [Lightkurve](https://docs.lightkurve.org/)

## Table of Contents
- [Overview](#overview)
- [A Hands-On Introduction](#a-hands-on-introduction)
- [Lightkurve Objects](#lightkurve-objects)
    * [The LightCurve Object](#the-lightcurve-object)
        - [Construction](#construction)
        - [Visualization](#visualization)
        - [Data Manipulation](#data-mainpulation)
    * [Target Pixel File Object](#target-pixel-file-object)
    * [Periodogram Objects](#periodogram-objects)
        - [Box Least Squares](#box-least-squares)
- [Exercises](#exercises)
    * [1. Secondary Eclipse of WASP-121 b](#1-secondary-eclipse-of-wasp-121-b)
    * [2. Exploring Transit Timing Variations](#2-exploring-transit-timing-variations-using-the-river-plot)
    * [3. Using Periodograms to Model Stellar Rotation](#3-using-periodograms-to-model-stellar-rotation)

# Overview
`Lightkurve` is a user-friendly Python package that is designed to make interacting with [TESS](https://science.nasa.gov/mission/tess/) and [Kepler](https://science.nasa.gov/mission/kepler/) time-series photometry easy and accessible. It is well documented, and there are pre-existing [tutorials](https://lightkurve.github.io/lightkurve/tutorials/index.html) that cover most functionality. This lesson will cover the primary use cases, best practices, and examples, along with exercises for hands-on learning and deeper understanding. 
**Run the following code to initialize the interactive portions of this lesson:**

from jupyterquiz import display_quiz
import warnings
import json
warnings.filterwarnings('ignore')
import os
q_dir = os.path.join('..', '..', 'Exercise_Solutions', 'Module_4', 'Lightkurve', 'Checkpoints', 'questions.json')
with open(q_dir, 'r') as file:
    questions = json.load(file)

# A Hands-On Introduction
There are three primary objects used in `Lightkurve`:
* `LightCurve` object
* `TargetPixelFile` objects
* `Periodogram` objects

Before diving into the details of these objects, this first section will give a quick start by getting you downloading, plotting, and visualizing an exoplanet transit. 

First, we import `lightkurve`:

import lightkurve as lk
from matplotlib import pyplot as plt
import numpy as np

The first step is to download the `target pixel file`, which is the 2D image used to construct the timeseries photometry. Let's investigate the known exoplanet system **WASP-121**:

## Downloads all pixel files for the target
pixelfile = lk.search_targetpixelfile("WASP-121").download_all()

Let's check out the first image.

##Check out different images by changing the index!
pixelfile[3].plot()
plt.show()

To see the light curve from this data, we use `pixelfile.to_lightcurve`:

##Select a target pixel to analyze, which contains one sector of timeseries data
##the .to_lightcurve method takes target pixels and converts them into integrated flux. The 'aperture_mask' 
##allows you to ignore other targets around the target of interest
lc = pixelfile[3].to_lightcurve(aperture_mask='all')

##Plot the results!
lc.plot()
plt.show()

We can see a repeated, deep transit signal here, but what if we want a better view? We can easily access different parts of the light curve by normal Python splicing:

## This plots the first 1400 data points in the light curve. 
lc[0:1400].plot()
plt.show()

Since this spans around ~0.5 days over 1400 data points, this data is derived from the TESS 2-minute cadence data from the [Quick Look Pipeline](https://tess.mit.edu/qlp/)! You can find more metadata by using the `LightCurve.meta` method:

##We can check the type to see which mission it's derived from, the target's TIC ID, which camera was used, during what sector, and the RA/DEC
print(lc.meta)
##This is just a dictionary, which can be queried by typical key:value syntax.
print('Target RA: ' + str(lc.meta['RA']), 'Target DEC: ' + str(lc.meta['DEC']))

## Checkpoint 1

Let's check our understanding so far! Run the codeblock below for some quiz questions.

display_quiz(questions[0:2])

Using the codeblock below, plot the **lightcurve** of the exoplanetary system HD 209458 via the **target pixel files**.

# Hint: Start off with downloading the TPFs with lk.search_targetpixelfile. Make sure you're looking at the right system (HD 209458)!


The next section will highlight the primary `LightCurve` methods, which involve phase folding, detrending, and data cleaning.

# [`Lightkurve` Objects](https://lightkurve.github.io/lightkurve/reference/index.html)

## [The LightCurve Object](https://lightkurve.github.io/lightkurve/reference/lightcurve.html)
This is the object we constructed above. The `Lightcurve` object requires, at a minimum, three parameters:
- Time (`time`)
- Flux (`flux`)
- Flux Error (`flux_err`)

### Construction
`LightCurves` can be constructed like above using `targetpixelfile` objects, or by passing three array-like objects to the `time`, `flux`, and `flux_err` parameters. We can retrieve these arrays by passing `LightCurve.time/flux/flux_err`. Using the example above:

#Retrieve the data columns
print(lc.time, lc.flux, lc.flux_err)

#Retrieve metadata
print(lc.meta)

### Visualization
`Lightkurve` provides four plotting routines. Since these routines are made using `matplotlib`, we can manipulate any plot with `matplotlib` methods. We can examine them below:

lc.plot()
plt.show()

lc.scatter()
plt.show()

lc.errorbar()
plt.show()

## River plots phase folds the light curve over a set period. For exoplanets, it essentially takes
## each transit and plots it on top of each other, which helps the SNR of the signal,
## Here, the transit manifests itself as a continuous dark band across the data at ~0.1 phase
lc.plot_river(period=1.2749255)
plt.show()

### Data Manipulation
To conduct research, we can apply a variety of methods to the light curve to clean and reconstruct it into a more useful form.

#### Detrending/Flattening
The lightcurves above are not perfectly flat, which may be due to systematic effects, host variability, or stellar rotation modulation by star spots. To mitigate this, we can use `LightCurve.flatten`, which applies a [Savitzky-Golay filter](https://en.wikipedia.org/wiki/Savitzky%E2%80%93Golay_filter) to remove these trends without affecting the short-term variation.

<div class="alert alert-block alert-info">
 
**IMPORTANT**: When detrending/flattening, the `window_length` parameter should be adjusted based on the time-scale of the underlying baseline variation. For example, the light curves above appear to vary throughout the observations. Thus, the window length should be well within the thousands of data points (~days).

##We need to change the default window_length here, which is 101 data points. This is too short a 
##timescale for our data, which varies over thousands of data points. Otherwise, we will detrend the
## the transits.
lc2 = lc.flatten(window_length=10001)
lc2.scatter()
plt.show()


##An example of bad flattening, even by using the defaults. We've lost our transits!
##Be wary of this. Check the original data to ensure you do not lose important signal, or use a mask, detailed below
lc3 = lc.flatten()
lc3.scatter()
plt.show()

## We can ignore transits using the 'create_transit_mask()' function, too. But this requires knowing the transit period, duration, and midpoint epoch
## These might be parameters you are trying to constrain!
tmask = lc.create_transit_mask(period = 1.27492550, duration = 0.120291667, transit_time = 2228.9)
lc10 = lc.flatten(window_length=101, mask=tmask)
lc10.scatter()
plt.show()

This flattened fairly well, while maintaining the depths of the transits. If there are shorter-frequency signals, you can also use the `mask` parameter in the flatten routine to make the routine ignore certain data points. The mask must be a Boolean array, where `True` data will not be considered in the detrending. This can be used to safely preserve the transits.

#### Clean-up Routines
We can use the `LightCurve.remove_nans` and `LightCurve.remove_outliers` routines to get rid of invalid or spurious data points. 

NOTE: `LightCurve.flatten` will automatically remove outliers, which can be adjusted using the `sigma` parameter. Additionally, it automatically normalizes the light curve, which has its own routine `LightCurve.normalize()`.

#Remove nans in-place, no need to redefine the variable
lc2 = lc2.remove_nans()
#Clip any spurious data points
lc2 = lc2.remove_outliers(sigma = 5)
lc2.scatter()
plt.show()

#### Phase-Folding and Binning
To get a better constraint on the shape of the transit, we can phase-fold and bin (essentially multi-point averages).

#The binning unit is assumed to be days. This bins every ~7 data points
lc2 = lc2.bin(time_bin_size=0.01)
lc2.scatter()
plt.show()

## We phase fold it based on the known epoch transit time, and set the mid transit phase to 0
lc2 = lc2.fold(period = 1.2749255, epoch_time=2456635.16, epoch_phase=0)
lc2.scatter()
plt.show()


This is a significant improvement! We can append more than one TESS sector by using the `LightCurve.append` methods and repeating the analysis as before:

## We can add more and more pixel files to the same light curve object. 
## These pixel files are not as long as the example above, so we use a smaller window_length
lc4 = (pixelfile[9].to_lightcurve()).flatten(window_length = 2001)
lc5 = (pixelfile[8].to_lightcurve()).flatten(window_length = 2001)
lc6 = (pixelfile[7].to_lightcurve()).flatten(window_length = 2001)
#...
lc7 = lc4.append(lc5)
lc7 = lc7.append(lc6)
#...
lc7 = lc7.bin(time_bin_size=0.01)
## Use the same phase folding parameters as before
lc7 = lc7.fold(period = 1.2749255, epoch_time=2456635.16, epoch_phase=0.65)
lc7.scatter()

plt.show()

### Export Methods
Lightcurves can be exported to a variety of different file types or other Python structures. These are outlined below:

##
#Send your lightcurve to a csv file
# lc.to_csv("Enter Path Here")

#To an Excel file
# lc.to_excel("Enter Path Here")

#To a fits file
# lc.to_fits("Enter Path Here")

#To a pandas dataframe
# lc.to_pandas("Enter Path Here")

#To an Astropy table
# lc.to_table("Enter Path Here")

Feel free to export your favorite graphs into any of the above file/structure formats!

## Checkpoint 2

That was a lot of information! Let's do another check to see how you use these tools! Run the codeblock below for a quick quiz question.

display_quiz(questions[2:3])

Adding to the codeblock below, let's plot some comparison graphs. First, plot the lightcurve as generated from the target pixel files. Then, flatten the lightcurve with an appropriate window and plot the flattened lightcurve. Remember that it should be long enough to remove long-term variations. Finally, fold the lightcurve on a period of 3.52474859 days. After folding the curve, bin the folded lightcurve to see the shape of the transit itself. Plot both the binned and unbinned versions, and play around with the `time_bin_size` parameter to compare. Remember that it uses the unit of days!

pixelfile2 = lk.search_targetpixelfile("HD 209458").download_all()
lc2 = pixelfile2[0].to_lightcurve(aperture_mask='all')
lc2.plot()

## [Target Pixel File Object](https://docs.lightkurve.org/reference/targetpixelfile.html)

The "raw" output of TESS and Kepler are target pixel files, which, as mentioned previously, are strung together to create time-series photometry. Normally, these are given as [.fits](https://fits.gsfc.nasa.gov/fits_primer.html) files, which are a standard, multi-dimensional data type that typically consists of header(s), units (i.e., metadata), and data units (i.e., time, flux). `Lightkurve` automatically downloads and extracts the relevant data from these `.fits` files in easy-to-read Python syntax. To see how to query your favorite systems and their corresponding data, check out the [tutorial](https://docs.lightkurve.org/tutorials/1-getting-started/searching-for-data-products.html).

This section goes over the basic methods of the `TargetPixelFile` object and best practices. This example covers Kepler target pixel files, but similar syntax also works for TESS.

tpf = lk.search_targetpixelfile('KIC 6922244', author="Kepler", quarter=4, cadence="long").download()

#Get mission data using the .meta property
for key, value in tpf.meta.items():
    print(f"{key}: {value}")

#Get the TPF dimensions
#This TPF has a dimensionality of 4116x5x5; that is, 4116, 5x5 images over time.
#If constructed to a light curve, it would yield 4116 data points
print(tpf.flux.shape)

#The resulting first image of the TPF
_ = tpf.plot(frame=0)
plt.show()

#We can get more metadata by using the .get_header property, too
for key, value in tpf.get_header()[0:10].items():
    print(f"{key}: {value}")

#And constructing the lightcurve
kepler_lc = tpf[0].to_lightcurve(aperture_mask='all')

We can animate the TPFs, too! 

Uncomment the code block below to generate the animation. Note that it may take 90 seconds or longer to execute.

# tpf.animate(1, 1)

As we saw in the [introduction](#a-hands-on-introduction), we can use these TPFs to construct custom light curves. You can see the rest of the related functions [here](https://docs.lightkurve.org/reference/targetpixelfile.html).

## `Periodogram` Objects

The last object in the `Lightkurve` module is periodograms, which are typically [Lomb-Scargle Periodograms](https://docs.astropy.org/en/stable/timeseries/lombscargle.html). You can think of this as a discrete Fast Fourier Transform, but it works with unevenly sampled data. The periodogram allows the identification of periodic signals in light curves, which can arise from transits, eclipsing binaries, stellar modulation, and systematics, to name a few. Here, we give a brief example of how it can be used to constrain the period of a transiting exoplanet.

## Begin with our initial light curve
lc_per = lc.flatten(window_length=5001)
lc_per.scatter()
plt.show()

periodogram = lc_per.to_periodogram(oversample_factor=1)
periodogram.plot()
plt.xlim(0, 10)
plt.show()

There are many peaks here! To get a better sense, we can plot in terms of period instead of frequency.

periodogram.plot(view='period', scale='log')
plt.show()

#Find the period with max power
print(periodogram.period_at_max_power)

lc.fold(period=periodogram.period_at_max_power).scatter()

plt.show()

But wait, that phase-folded light curve doesn't look right! And in fact, that is not the period of WASP121 b at all. What gives? Well, the other peaks with high power in the periodogram are *harmonics*. Careful observation shows that the max power period above is really the $\frac{period}{2}$. Trying this and we get:

lc_per.fold(period=periodogram.period_at_max_power*2).scatter()
plt.show()

Better, but not great... This leads to an important note when using periodograms:

<div class="alert alert-block alert-info">
 
**IMPORTANT**: Periodograms are discretely binned. The precision of the extracted period depends on the binning resolution. This can be improved by increasing the `over_sample_factor`. Do note that the higher this factor, the more computationally intensive it will be. 

periodogram = lc_per.to_periodogram(oversample_factor=100)
periodogram.plot(view='period', scale='log')
period = (periodogram.period_at_max_power).value*2
lc_per = lc_per.fold(period=period)
lc_per = lc_per.bin(time_bin_size=0.001)
lc_per.scatter()
plt.show()

A significant improvement! We can then use this to derive the **period** and the **duration** to model our primitive transit.

### Box-Least Squares
A basic model to fit transiting exoplanets is the `Box-Least Squares` routine. This model is simple: it is a flat line centered at normalized flux 1, with a transit modeled as a box with a given `period`, `duration`, and `depth`. Luckily, we only need to provide the first two parameters, for which we have decent initial guesses. Since it relies on a *periodic signal*, we need to apply it to our flattened, but **not** phase-folded light curve.

BLS = lk.periodogram.BoxLeastSquaresPeriodogram.from_lightcurve(lc)
model = BLS.get_transit_model(period = period, transit_time= 2228.908, duration=0.1)
model = model.normalize()
ax = lc.flatten(window_length=10001)
ax = ax.normalize()
ax = ax.fold(float(period), 2228.9)
ax = ax.bin(time_bin_size=0.001).scatter()
model.fold(float(period), 2228.9).plot(ax=ax, c='r', lw=2)
plt.show()

The model fits quite well! We notice that the baseline is higher than 1.0 normalized flux in our data; this is due to the depth of the transit. This tends to ***overestimate*** the true quiescent flux from the host because the transit biases the median downward. This can be corrected, e.g., using the flux during the *secondary eclipse*.

## Checkpoint 3

We've finally gotten through the module; before moving on to the Exercises, let's have one last checkpoint. Run the following codeblock for quiz questions.

display_quiz(questions[3:5])

Building on the previous checkpoint, use the flattened lightcurve for HD 209458, generate and plot a BLS Periodogram for the lightcurve. Then fold the light curve using the period from the Periodogram. Compare with the folded curve using the published value of 3.52474859 days  (found in [Bonomo et al., 2017](https://arxiv.org/abs/1704.00373)). Was your periodogram fine enough? How do you know? What could you change it to get a closer value?

# You don't have to run the following lines if you've already completed Checkpoint 2 - the variables are already defined!

# pixelfile2 = lk.search_targetpixelfile("HD 209458").download_all()
# lc2 = pixelfile2[0].to_lightcurve(aperture_mask='all')

# Exercises 

## 1. Secondary Eclipse of Tylos (WASP-121 b)
[Tylos (WASP-121 b)](https://svs.gsfc.nasa.gov/31272/) is a known **ultrahot** Jupiter, having an equilibrium temperature of $\sim 2600 \; \mathrm{K}$. This means it emits and reflects an appreciable amount of light. When the planet is away from our line of sight, there is actually a **second transit**, where the brightness of the system drops since the planet no longer contributes to the quiescent flux. To differentiate it from the primary transit, the secondary transit is sometimes called an **occultation** or a **secondary eclipse**. The period is necessarily identical to the primary, but with a phase offset of $\frac{period}{2}$. For this exercise, your goal is to:
* Append at least five TESS visits to a single light curve
* Detrend/flatten and clean up the data
* Phase fold all visits into one phase curve, **centered** on the secondary eclipse
* Use the box least squares periodogram to model and overlay the model on your phase-folded data

HINT: When searching for light curves, you can use the compound method `search_lightcurve().download_all().stitch()` to derive all light curves into one light curve object.

[Sample Solution](../../Exercise_Solutions/Module_4/Lightkurve/Exercises/Exercise_1.ipynb)

In [ ]:
#Codespace for Exercise 1

## 2. Exploring Transit Timing Variations Using the River Plot
As briefly demonstrated, river plots are useful 2D plots that can help identify periodic signals. [Transit Timing Variations (TTVs)](https://lco.global/science/projects/transit-timing-variations-ttv/#:~:text=The%20transits%20of%20a%20planet,Murray%202005%2C%20Agol%20et%20al.) are changes in the timing of planet transits that can occur in multi-planetary or multi-stellar systems due to the complex dynamics of the system. TTVs can encode rich information about these dynamics, but they can be difficult to analyze, since the period varies on short time scales. Luckily, the river plot method can be a great investigative tool for this. [**KOI-227**](https://www.google.com/search?q=koi+227&rlz=1C1CHBF_enUS910US910&oq=koi+227&gs_lcrp=EgZjaHJvbWUyBggAEEUYOdIBCDIxNjFqMWo3qAIAsAIA&sourceid=chrome&ie=UTF-8), or **KIC 6185476**, is known to exhibit TTVs. In this exercise:
* Query the **long cadence** TPFs for **KIC 6185476**, which are derived from the Kepler mission, and combine them into one dataset
* Detrend the light curve appropriately (This may take some trial and error)
* Use the period and transit epoch entries from the [NASA Exoplanet Archive](https://exoplanetarchive.ipac.caltech.edu/) to phase fold. Comment on what you observe.
* Make a river plot of the folded light curve

[Sample Solution](../../Exercise_Solutions/Module_4/Lightkurve/Exercises/Exercise_2.ipynb)

In [ ]:
#Codespace for Exercise 2

## 3. Using Periodograms to Model Stellar Rotation
**Star spots** on stars are cool regions, analogous to sunspots, that can cause sinusoidal variations in a star's light curve as it rotates, where dips are observed when the star spot is in our line of sight, and peaks when it is outside our line of sight. These can be modeled using the Lomb-Scargle periodogram, similar to the BLS method enumerated throughout this lesson. **KIC 2157356** is an exoplanetary system that is known to have a strong stellar rotation signal. Using a similar heuristic to the BLS method, model the rotation signal to find how quickly the star rotates by:
* Downloading and stitching together all available light curves from Kepler
* Generating a periodogram of the light curve, using the `maximum_period = 100` parameter
* Derive the period with the maximum power, and use this to generate a periodogram model with the `periodogram.model` method
* While this is not directly related to exoplanets, it can be important to model the rotation of stars to **remove** the rotation signal, which can otherwise confound the BLS transit model.

[Sample Solution](../../Exercise_Solutions/Module_4/Lightkurve/Exercises/Exercise_3.ipynb)

In [ ]:
#Codespace for Exercise 3